In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("tra/Users/tanjimhossain/Documents/machine-learning-project/nfl-big-data-bowl-2026-prediction/trainin")

input_w01  = pd.read_csv(DATA_DIR / "input_2023_w01.csv")
output_w01 = pd.read_csv(DATA_DIR / "output_2023_w01.csv")

print(input_w01.shape, output_w01.shape)
print(input_w01.head())
print(output_w01.head())


ModuleNotFoundError: No module named 'pandas'

In [2]:
# last pre-pass frame per player
throw_state = (
    input_w01
    .sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
    .groupby(["game_id", "play_id", "nfl_id"], as_index=False)
    .tail(1)          # last frame before throw
)

throw_state = throw_state.rename(columns={"frame_id": "frame_id_input"})


In [3]:
throw_state["num_frames_output"].describe()
throw_state["player_role"].value_counts()
throw_state["player_to_predict"].value_counts()


player_to_predict
False    7410
True     2679
Name: count, dtype: int64

In [4]:
train_w01 = output_w01.merge(
    throw_state,
    on=["game_id", "play_id", "nfl_id"],
    how="inner",
    suffixes=("_future", "_throw")
)

print(train_w01.shape)
train_w01.head()


(32088, 26)


,game_id,play_id,nfl_id,frame_id,x_future,y_future,player_to_predict,frame_id_input,play_direction,absolute_yardline_number,...,player_role,x_throw,y_throw,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y
0,2023090700,101,46137,1,56.22,17.28,True,26,right,42,...,Defensive Coverage,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22
1,2023090700,101,46137,2,56.63,16.88,True,26,right,42,...,Defensive Coverage,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22
2,2023090700,101,46137,3,57.06,16.46,True,26,right,42,...,Defensive Coverage,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22
3,2023090700,101,46137,4,57.48,16.02,True,26,right,42,...,Defensive Coverage,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22
4,2023090700,101,46137,5,57.91,15.56,True,26,right,42,...,Defensive Coverage,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22


In [8]:
# inspect future frame index vs number of frames to predict
train_w01[["frame_id", "num_frames_output"]].describe()

# this one is already correct:
train_w01[["x_future", "y_future"]].describe()


,x_future,y_future
count,32088.000000,32088.000000
mean,59.020316,26.530208
std,25.550084,13.249616
min,0.420000,0.590000
25%,40.567500,15.050000
50%,58.430000,26.190000
75%,77.000000,38.080000
max,119.380000,52.910000


In [9]:
# Every future frame_id should be within the num_frames_output
(train_w01["frame_id"] <= train_w01["num_frames_output"]).value_counts()

# How many future frames per player_to_predict?
train_w01.groupby("player_to_predict")["frame_id"].max().describe()


count     1.0
mean     94.0
std       NaN
min      94.0
25%      94.0
50%      94.0
75%      94.0
max      94.0
Name: frame_id, dtype: float64

In [10]:
import numpy as np

def normalize_direction(df):
    df = df.copy()
    mask = df["play_direction"] == "left"

    # Flip x and y for throw & future + ball landing
    df.loc[mask, "x_throw"]   = 120 - df.loc[mask, "x_throw"]
    df.loc[mask, "y_throw"]   = 53.3 - df.loc[mask, "y_throw"]
    df.loc[mask, "x_future"]  = 120 - df.loc[mask, "x_future"]
    df.loc[mask, "y_future"]  = 53.3 - df.loc[mask, "y_future"]
    df.loc[mask, "ball_land_x"] = 120 - df.loc[mask, "ball_land_x"]
    df.loc[mask, "ball_land_y"] = 53.3 - df.loc[mask, "ball_land_y"]

    # Also flip orientation / direction by 180°
    df.loc[mask, "dir"] = (df.loc[mask, "dir"] + 180) % 360
    df.loc[mask, "o"]   = (df.loc[mask, "o"] + 180) % 360

    df["play_direction"] = "right"
    return df

train_w01n = normalize_direction(train_w01)


In [11]:
import numpy as np

train = train_w01n.copy()

# Time since throw (seconds) – frame_id in output is 1..num_frames_output, 10 fps
train["t"] = train["frame_id"] / 10.0

# Velocity components at throw (dir is in degrees)
train["dir_rad"] = np.deg2rad(train["dir"])
train["vx0"] = train["s"] * np.cos(train["dir_rad"])
train["vy0"] = train["s"] * np.sin(train["dir_rad"])

# Constant-velocity predicted positions
train["x_pred_cv"] = train["x_throw"] + train["vx0"] * train["t"]
train["y_pred_cv"] = train["y_throw"] + train["vy0"] * train["t"]

# Compute RMSE only on players that are scored
mask = train["player_to_predict"]

rmse_cv = np.sqrt(
    np.mean(
        (train.loc[mask, "x_future"] - train.loc[mask, "x_pred_cv"])**2 +
        (train.loc[mask, "y_future"] - train.loc[mask, "y_pred_cv"])**2
    )
)
print("Constant-velocity RMSE (week 1):", rmse_cv)


Constant-velocity RMSE (week 1): 9.285200502823058


In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("train")

def get_throw_state(input_df: pd.DataFrame) -> pd.DataFrame:
    throw_state = (
        input_df
        .sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
        .groupby(["game_id", "play_id", "nfl_id"], as_index=False)
        .tail(1)
        .rename(columns={"frame_id": "frame_id_input"})
    )
    return throw_state

def normalize_direction(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    mask = df["play_direction"] == "left"

    df.loc[mask, "x_throw"]   = 120 - df.loc[mask, "x_throw"]
    df.loc[mask, "y_throw"]   = 53.3 - df.loc[mask, "y_throw"]
    df.loc[mask, "x_future"]  = 120 - df.loc[mask, "x_future"]
    df.loc[mask, "y_future"]  = 53.3 - df.loc[mask, "y_future"]
    df.loc[mask, "ball_land_x"] = 120 - df.loc[mask, "ball_land_x"]
    df.loc[mask, "ball_land_y"] = 53.3 - df.loc[mask, "ball_land_y"]

    df.loc[mask, "dir"] = (df.loc[mask, "dir"] + 180) % 360
    df.loc[mask, "o"]   = (df.loc[mask, "o"] + 180) % 360

    df["play_direction"] = "right"
    return df

def build_week_dataset(week: int) -> pd.DataFrame:
    input_df  = pd.read_csv(DATA_DIR / f"input_2023_w{week:02d}.csv")
    output_df = pd.read_csv(DATA_DIR / f"output_2023_w{week:02d}.csv")

    throw_state = get_throw_state(input_df)

    week_df = output_df.merge(
        throw_state,
        on=["game_id", "play_id", "nfl_id"],
        how="inner",
        suffixes=("_future", "_throw"),
    )

    # rename for clarity
    week_df = week_df.rename(columns={"x_future": "x_future",
                                      "y_future": "y_future"})
    week_df = normalize_direction(week_df)

    return week_df


In [13]:
all_weeks = [build_week_dataset(w) for w in range(1, 19)]
train_all = pd.concat(all_weeks, ignore_index=True)
print(train_all.shape)


(562936, 26)


In [14]:
train_all = train_all.copy()
train_all["t"] = train_all["frame_id"] / 10.0

train_all["dir_rad"] = np.deg2rad(train_all["dir"])
train_all["vx0"] = train_all["s"] * np.cos(train_all["dir_rad"])
train_all["vy0"] = train_all["s"] * np.sin(train_all["dir_rad"])

train_all["x_pred_cv"] = train_all["x_throw"] + train_all["vx0"] * train_all["t"]
train_all["y_pred_cv"] = train_all["y_throw"] + train_all["vy0"] * train_all["t"]

mask = train_all["player_to_predict"]
rmse_cv_all = np.sqrt(
    np.mean(
        (train_all.loc[mask, "x_future"] - train_all.loc[mask, "x_pred_cv"])**2 +
        (train_all.loc[mask, "y_future"] - train_all.loc[mask, "y_pred_cv"])**2
    )
)
print("Constant-velocity RMSE (all weeks):", rmse_cv_all)


Constant-velocity RMSE (all weeks): 8.163254398161516


In [15]:
train_all["dx_future"] = train_all["x_future"] - train_all["x_throw"]
train_all["dy_future"] = train_all["y_future"] - train_all["y_throw"]

# Ball-relative features
train_all["dist_ball"] = np.sqrt(
    (train_all["ball_land_x"] - train_all["x_throw"])**2 +
    (train_all["ball_land_y"] - train_all["y_throw"])**2
)
train_all["angle_to_ball"] = np.arctan2(
    train_all["ball_land_y"] - train_all["y_throw"],
    train_all["ball_land_x"] - train_all["x_throw"],
)


In [16]:
feature_cols = [
    "t",
    "x_throw", "y_throw",
    "s", "a",
    "vx0", "vy0",
    "dist_ball", "angle_to_ball",
    "num_frames_output",
]


In [17]:
train_ml = train_all[train_all["player_to_predict"]].reset_index(drop=True)


In [18]:
train_ml[feature_cols + ["dx_future", "dy_future"]].describe()


,t,x_throw,y_throw,s,a,vx0,vy0,dist_ball,angle_to_ball,num_frames_output,dx_future,dy_future
count,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000
mean,0.786540,67.020923,26.491414,4.811613,2.732318,-0.055878,2.876556,11.778709,-0.006234,14.730804,2.486389,-0.000334
std,0.592385,22.991527,11.787700,2.052455,1.465647,3.508296,2.603534,7.443833,1.543187,7.160791,3.991773,3.728389
min,0.100000,7.440000,0.690000,0.000000,0.000000,-10.049202,-6.342470,0.022361,-3.141593,5.000000,-13.130000,-22.780000
25%,0.400000,48.510000,16.250000,3.300000,1.580000,-2.662554,1.013957,6.263547,-1.300621,10.000000,0.240000,-1.760000
50%,0.700000,64.520000,26.300000,4.770000,2.550000,-0.031303,2.753296,9.762234,-0.000677,13.000000,1.110000,-0.010000
75%,1.100000,84.440000,36.800000,6.350000,3.730000,2.543443,4.658872,16.037687,1.262041,19.000000,3.210000,1.740000
max,9.400000,119.390000,52.620000,10.340000,8.360000,9.589657,10.309109,49.740375,3.141593,94.000000,31.050000,23.380000


In [19]:
cols_to_check = feature_cols + ["dx_future", "dy_future"]
print(train_ml[cols_to_check].isnull().sum())


t                    0
x_throw              0
y_throw              0
s                    0
a                    0
vx0                  0
vy0                  0
dist_ball            0
angle_to_ball        0
num_frames_output    0
dx_future            0
dy_future            0
dtype: int64


In [20]:
from sklearn.model_selection import train_test_split
import numpy as np

# Unique games
game_ids = train_ml["game_id"].unique()

train_games, val_games = train_test_split(
    game_ids,
    test_size=0.2,
    random_state=42,
)

train_mask = train_ml["game_id"].isin(train_games)
val_mask   = train_ml["game_id"].isin(val_games)

X_train = train_ml.loc[train_mask, feature_cols]
X_val   = train_ml.loc[val_mask,   feature_cols]

y_train_dx = train_ml.loc[train_mask, "dx_future"]
y_train_dy = train_ml.loc[train_mask, "dy_future"]
y_val_dx   = train_ml.loc[val_mask,   "dx_future"]
y_val_dy   = train_ml.loc[val_mask,   "dy_future"]

len(X_train), len(X_val)


(445478, 117458)

In [21]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

rf_dx = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
)
rf_dy = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
)

rf_dx.fit(X_train, y_train_dx)
rf_dy.fit(X_train, y_train_dy)

dx_pred_val = rf_dx.predict(X_val)
dy_pred_val = rf_dy.predict(X_val)

# RMSE in displacement space (same as in x,y space)
rmse_model = np.sqrt(
    np.mean((y_val_dx - dx_pred_val)**2 + (y_val_dy - dy_pred_val)**2)
)
print("RandomForest RMSE (val):", rmse_model)


RandomForest RMSE (val): 1.7667896329596222


In [22]:
# Indices of validation rows (in train_ml / train_all)
val_rows = train_ml.loc[val_mask].index

x_true = train_all.loc[val_rows, "x_future"]
y_true = train_all.loc[val_rows, "y_future"]

x_cv   = train_all.loc[val_rows, "x_pred_cv"]
y_cv   = train_all.loc[val_rows, "y_pred_cv"]

rmse_cv_val = np.sqrt(np.mean((x_true - x_cv)**2 + (y_true - y_cv)**2))
print("Constant-velocity RMSE (val):", rmse_cv_val)
print("RandomForest RMSE (val):", rmse_model)
print("Improvement factor:", rmse_cv_val / rmse_model)


Constant-velocity RMSE (val): 8.712156687660606
RandomForest RMSE (val): 1.7667896329596222
Improvement factor: 4.931066226071586


In [23]:
import pandas as pd

fi_dx = pd.Series(rf_dx.feature_importances_, index=feature_cols).sort_values(ascending=False)
fi_dy = pd.Series(rf_dy.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("Feature importance for dx:")
print(fi_dx)

print("\nFeature importance for dy:")
print(fi_dy)


Feature importance for dx:
t                    0.558983
vy0                  0.345312
angle_to_ball        0.037144
dist_ball            0.018623
num_frames_output    0.010411
a                    0.009650
y_throw              0.007189
vx0                  0.004484
s                    0.004346
x_throw              0.003858
dtype: float64

Feature importance for dy:
vx0                  0.605963
t                    0.231469
angle_to_ball        0.107688
y_throw              0.015286
a                    0.010209
dist_ball            0.009743
s                    0.006446
vy0                  0.005142
x_throw              0.004161
num_frames_output    0.003894
dtype: float64
